# **PIPELINE COMPLETO: DE DATOS A MODELO DESPLEGADO**

> **Versión comentada para alumnos.** Se conserva el flujo y la solución del Hands On oficial; los comentarios adicionales explican qué hace cada bloque, qué conceptos reutiliza de clases anteriores y qué deben observar durante la ejecución.


Este Colab conecta en un solo flujo lo visto en los tres Temas anteriores (inferencia básica, RAG y fine-tuning con LoRa) para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico. Es la base directa para el reto del Hackathon 1.

**Nota de alcance:** "Desplegar" aquí significa guardar el modelo ajustado de forma reutilizable y envolverlo en una función lista para usar, no se levanta un servidor ni un endpoint público.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [ ]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================
#
# Este Hands On integra RAG + LoRA + una función de asistente.
# Por eso necesitamos librerías de:
#
# - transformers: cargar modelos y tokenizers.
# - peft: aplicar LoRA.
# - accelerate: facilitar el uso del hardware disponible.
# - trl: realizar Supervised Fine-Tuning con SFTTrainer.
# - sentence-transformers: generar embeddings para RAG.
#
# También utilizaremos un token de Hugging Face almacenado
# en Colab Secrets.
#
# ANTES DE EJECUTAR:
# 1. En Colab abre Entorno de ejecución > Cambiar tipo de entorno.
# 2. Selecciona GPU.
# 3. En el ícono de llave (Secrets), crea:
#       HF_TOKEN
# 4. Pega tu token de Hugging Face.
# 5. Activa el acceso del notebook al secreto.
#
# Más adelante, la parte final también requerirá:
#       GROQ_API_KEY
# guardada de la misma forma en Colab Secrets.

!pip install transformers peft accelerate trl sentence-transformers --quiet

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

# Reducimos mensajes internos de Transformers para que la salida sea más limpia.
logging.set_verbosity_error()

# Recuperamos el token desde Colab Secrets.
login(token=userdata.get('HF_TOKEN'))

print("Sesión de Hugging Face iniciada correctamente.")
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 33.6 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.
GPU disponible: True
GPU: Tesla T4


## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Indexamos una base de conocimiento propia con `sentence-transformers` para poder recuperar el fragmento más relevante antes de responder, esto es lo que evita que el asistente alucine sobre políticas que no conoce.

In [ ]:
# ============================================================
# PASO 1A. CREAR LA BASE DE CONOCIMIENTO PARA RAG
# ============================================================
#
# RAG necesita una fuente externa de información.
# En este ejemplo, esa fuente es un pequeño reglamento de biblioteca.
#
# Cada elemento de la lista "documentos" funciona como un fragmento
# independiente que posteriormente podrá recuperarse.
#
# Después usamos SentenceTransformer para transformar cada fragmento
# en un embedding: una representación numérica de su significado.
#
# Este procedimiento reutiliza lo visto en la Clase 2.

from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo multilingüe para crear embeddings.
modelo_embeddings = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

documentos = [
    "Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.",
    "Las devoluciones tardías generan una multa de $10 por día de retraso, hasta un "
    "máximo de $150 por libro.",
    "Los libros de la sección de reserva (material de examen) solo se consultan dentro de la biblioteca, no se prestan a domicilio."
]

# Convertimos los tres textos en vectores numéricos.
embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

# Observa:
# La primera dimensión corresponde al número de documentos.
# La segunda corresponde al tamaño del vector de embedding.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# ============================================================
# PASO 1B. CREAR LA FUNCIÓN DE RECUPERACIÓN
# ============================================================
#
# La función buscar_fragmento() implementa la parte "Retrieval" de RAG.
#
# Flujo:
# pregunta
#    ↓
# embedding de la pregunta
#    ↓
# comparación con embeddings de los documentos
#    ↓
# seleccionar el índice con mayor puntuación
#    ↓
# devolver el fragmento correspondiente
#
# IMPORTANTE:
# El notebook oficial utiliza np.dot(...) para obtener la puntuación
# de similitud. Conservamos exactamente ese procedimiento.

def buscar_fragmento(pregunta):

    # Convertimos la pregunta a embedding.
    embedding_pregunta = modelo_embeddings.encode([pregunta])

    # Calculamos una puntuación entre la pregunta y cada documento.
    similitudes = np.dot(
        embeddings_documentos,
        embedding_pregunta.T
    ).flatten()

    # np.argmax devuelve la posición de la puntuación más alta.
    indice_mas_similar = np.argmax(similitudes)

    # Recuperamos el texto original ubicado en esa posición.
    return documentos[indice_mas_similar]


# Prueba rápida del sistema de recuperación.
fragmento_prueba = buscar_fragmento(
    "¿Puedo llevarme a casa un libro de la sección de reserva?"
)

print("Fragmento recuperado:")
print(fragmento_prueba)

# Pregunta para reflexionar:
# ¿Por qué este fragmento es más útil que enviar todos los documentos
# al modelo en cada consulta?


Fragmento recuperado:
Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.


## **PASO 2: AJUSTAR EL TONO DEL MODELO (FINE-TUNING CON LoRA)**

Cargamos un modelo ligero basado en Llama y lo ajustamos con LoRA para que responda siempre en el mismo tono breve y directo, la configuración de esta celda ya fue validada en el tema 3.

In [ ]:
# ============================================================
# PASO 2A. CARGAR EL MODELO BASE
# ============================================================
#
# El Hands On utiliza TinyLlama como modelo ligero para demostrar
# el proceso de fine-tuning con LoRA en Google Colab.
#
# tokenizer:
# convierte texto en tokens y tokens nuevamente en texto.
#
# modelo:
# contiene los parámetros del modelo de lenguaje.
#
# dtype=torch.float16:
# reduce el uso de memoria al cargar el modelo.
#
# device_map="auto":
# permite a Transformers ubicar el modelo automáticamente
# en el dispositivo disponible.

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    dtype=torch.float16,
    device_map="auto"
)

print("Modelo base cargado:", modelo_base)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
# ============================================================
# PASO 2B. CREAR UNA FUNCIÓN PARA GENERAR RESPUESTAS
# ============================================================
#
# Esta función nos permitirá probar el modelo antes y después
# del fine-tuning utilizando la misma interfaz.
#
# TinyLlama es un modelo tipo chat, por eso usamos:
# tokenizer.apply_chat_template(...)
#
# Esa función convierte nuestra lista de mensajes al formato
# conversacional esperado por el modelo.

def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):

    # Creamos un mensaje con rol "user".
    mensajes = [
        {
            "role": "user",
            "content": pregunta
        }
    ]

    # Aplicamos la plantilla de chat del tokenizer.
    prompt_formateado = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convertimos el prompt a tensores y los movemos al dispositivo del modelo.
    entrada = tokenizer(
        prompt_formateado,
        return_tensors="pt"
    ).to(modelo_a_usar.device)

    # Generamos la continuación.
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )

    # La salida incluye también los tokens del prompt.
    # Aquí conservamos solamente los tokens nuevos generados.
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]

    # Convertimos esos tokens nuevamente a texto.
    return tokenizer.decode(
        tokens_nuevos,
        skip_special_tokens=True
    ).strip()


# Esta pregunta se utilizará como referencia antes del ajuste.
prompt_prueba = "¿Puedo renovar un préstamo?"


In [ ]:
# ============================================================
# PASO 2C. CONSTRUIR EL DATASET DE FINE-TUNING
# ============================================================
#
# Ahora preparamos ejemplos de pregunta/respuesta.
#
# IMPORTANTE:
# La base de conocimiento RAG y el dataset LoRA cumplen funciones
# diferentes.
#
# RAG:
#   aporta información externa al momento de responder.
#
# LoRA:
#   ajusta el comportamiento del modelo utilizando ejemplos.
#
# formatear_ejemplo() utiliza la plantilla chat de TinyLlama para
# preparar correctamente cada par usuario/asistente.

from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):

    mensajes = [
        {
            "role": "user",
            "content": pregunta
        },
        {
            "role": "assistant",
            "content": respuesta
        }
    ]

    return tokenizer.apply_chat_template(
        mensajes,
        tokenize=False
    )


pares = [
    (
        "¿Puedo renovar un préstamo?",
        "Sí, puedes renovarlo una vez si nadie más lo ha solicitado."
    ),
    (
        "¿Hasta cuándo puedo tener un libro prestado?",
        "El préstamo estándar es de 15 días."
    ),
    (
        "¿Necesito credencial para entrar a la biblioteca?",
        "Sí, es necesario mostrar tu credencial vigente en la entrada."
    ),
    (
        "¿Puedo reservar una sala de estudio?",
        "Sí, puedes reservarla hasta con 2 días de anticipación desde el portal."
    ),
    (
        "¿Hay wifi disponible dentro de la biblioteca?",
        "Sí, la red 'Biblioteca-Invitados' está disponible en todas las salas."
    ),
]

# Transformamos cada par en un texto con formato de chat.
ejemplos = [
    {
        "texto": formatear_ejemplo(p, r)
    }
    for p, r in pares
]

# Creamos el Dataset que utilizará SFTTrainer.
dataset = Dataset.from_list(ejemplos)

dataset


Dataset({
    features: ['texto'],
    num_rows: 5
})

In [ ]:
# ============================================================
# PASO 2D. CREAR UNA LÍNEA BASE
# ============================================================
#
# Probamos el modelo ANTES del ajuste.
#
# Esto sirve para tener una referencia cualitativa y después
# observar si LoRA modificó su comportamiento.

respuesta_base = generar_respuesta(
    modelo,
    prompt_prueba
)
#prompt_prueba = "¿Puedo renovar un préstamo?"
print("Respuesta del modelo base:")
print(respuesta_base)

# Guarda esta respuesta para compararla con el modelo ajustado.


Respuesta del modelo base:
Sí, puedes renovar el préstemo si deseas. La renovación de un préstergo es una operación de reembolso de la cantidad de la inversión original, pero con una nueva fecha de retención. En este caso, debes


In [ ]:
# ============================================================
# PASO 2E. CONFIGURAR LoRA
# ============================================================
#
# Esta configuración reutiliza lo visto en la Clase 3.
#
# r=8
#   Rango de las matrices LoRA.
#
# lora_alpha=16
#   Factor de escala de la actualización introducida por LoRA.
#
# target_modules=["q_proj", "v_proj"]
#   Módulos del mecanismo de atención donde se insertan adapters.
#
# lora_dropout=0.0
#   En esta demostración no se aplica dropout a LoRA.
#
# task_type="CAUSAL_LM"
#   Indica que estamos ajustando un modelo causal de lenguaje.
#
# La idea central:
# el modelo base NO se reentrena por completo.
# Solo se entrenan pequeñas matrices adicionales.

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed

# Semilla fija para favorecer reproducibilidad en la demostración.
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

# Aplicamos LoRA sobre TinyLlama.
modelo_lora = get_peft_model(
    modelo,
    config_lora
)

# Muestra cuántos parámetros son realmente entrenables.
modelo_lora.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# ============================================================
# PASO 2F. ENTRENAR LOS ADAPTERS LoRA
# ============================================================
#
# SFTTrainer realiza Supervised Fine-Tuning sobre el Dataset.
#
# Parámetros principales:
#
# num_train_epochs=30
#   El pequeño dataset se recorre varias veces.
#
# per_device_train_batch_size=5
#   En esta demo entran los cinco ejemplos en un lote.
#
# learning_rate=2e-4
#   Tamaño de las actualizaciones realizadas durante entrenamiento.
#
# logging_steps=1
#   Registramos el loss en cada paso.
#
# dataset_text_field="texto"
#   Columna que contiene los ejemplos ya formateados.
#
# max_length=128
#   Longitud máxima usada durante preparación del texto.

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

# Aquí ocurre el entrenamiento real.
resultado_entrenamiento = trainer.train()

# El notebook oficial compara:
# - el primer loss registrado;
# - el loss del último paso registrado.
#
# PRECISIÓN:
# perdida_inicial NO representa una evaluación independiente
# previa al entrenamiento; es el primer loss guardado en log_history.

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss']  # último paso del entrenamiento

# Alternativa incluida en el notebook oficial:
# perdida_final = resultado_entrenamiento.training_loss
# Esa opción representa el promedio reportado del entrenamiento.

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")

if perdida_inicial != 0:
    print(
        f"Reducción: {(1 - perdida_final / perdida_inicial) * 100:.0f}%"
    )

# Una reducción del loss indica que el modelo se ajustó mejor
# al patrón de estos datos.
#
# NO significa que el modelo sea mejor para cualquier tarea.


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.808', 'grad_norm': '2.961', 'learning_rate': '0.0002', 'entropy': '1.368', 'num_tokens': '251', 'mean_token_accuracy': '0.5976', 'epoch': '1'}
{'loss': '2.748', 'grad_norm': '2.762', 'learning_rate': '0.0001933', 'entropy': '1.372', 'num_tokens': '502', 'mean_token_accuracy': '0.6016', 'epoch': '2'}
{'loss': '2.657', 'grad_norm': '2.782', 'learning_rate': '0.0001867', 'entropy': '1.377', 'num_tokens': '753', 'mean_token_accuracy': '0.6016', 'epoch': '3'}
{'loss': '2.553', 'grad_norm': '2.799', 'learning_rate': '0.00018', 'entropy': '1.379', 'num_tokens': '1004', 'mean_token_accuracy': '0.6016', 'epoch': '4'}
{'loss': '2.441', 'grad_norm': '2.887', 'learning_rate': '0.0001733', 'entropy': '1.388', 'num_tokens': '1255', 'mean_token_accuracy': '0.6016', 'epoch': '5'}
{'loss': '2.322', 'grad_norm': '3.036', 'learning_rate': '0.0001667', 'entropy': '1.399', 'num_tokens': '1506', 'mean_token_accuracy': '0.6057', 'epoch': '6'}
{'loss': '2.207', 'grad_norm': '3.215', 'learning_rate

## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [ ]:
# ============================================================
# PASO 3. GUARDAR EL ADAPTADOR LoRA
# ============================================================
#
# En ESTE Hands On, "desplegar" no significa crear un servidor
# ni publicar un endpoint.
#
# Significa guardar el adapter entrenado para poder reutilizarlo
# posteriormente sin repetir todo el entrenamiento.
#
# save_pretrained(...) guarda los archivos del adapter localmente.

modelo_lora.save_pretrained(
    "/content/modelo_ajustado_lora"
)

print(
    "Adaptador LoRA guardado en /content/modelo_ajustado_lora"
)

# OPCIONAL:
# Si quisieras conservarlo en tu cuenta de Hugging Face:
#
# modelo_lora.push_to_hub(
#     "tu-usuario/tinyllama-atencion-clientes-lora"
# )
#
# Esta línea está comentada porque no es necesaria para completar
# el Hands On.


Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: EL PIPELINE COMPLETO — RAG + MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

***Nota:** TinyLlama aprendió un tono específico con solo 5 ejemplos, suficiente para demostrar que LoRA funciona, pero no para generalizar a una instrucción nueva con contexto inyectado, como la que pide este pipeline con RAG.*

*Por lo que, un modelo real desplegado sí tendría esa capacidad, usando aquí Groq (GPT-OSS-20B) en lugar del modelo ajustado con LoRA como muestra de ese comportamiento.*

In [ ]:
# ============================================================
# PASO 4. CONECTAR EL PIPELINE: RETRIEVAL + GENERACIÓN
# ============================================================
#
# Esta es la parte donde se conectan las piezas.
#
# Flujo de la función asistente():
#
# pregunta del usuario
#         ↓
# buscar_fragmento()
#         ↓
# fragmento relevante del reglamento
#         ↓
# construir prompt con contexto
#         ↓
# enviar prompt al modelo generativo
#         ↓
# devolver respuesta
#
# IMPORTANTE:
# Aunque en pasos anteriores entrenamos TinyLlama + LoRA,
# ESTA FUNCIÓN FINAL utiliza Groq con:
#
#     openai/gpt-oss-20b
#
# El propio notebook oficial explica por qué:
# cinco ejemplos bastan para demostrar el mecanismo de LoRA,
# pero no para esperar que TinyLlama generalice correctamente
# a una nueva instrucción RAG con contexto inyectado.
#
# Por eso el modelo de Groq se utiliza aquí como demostración
# del comportamiento final del pipeline.
#
# Antes de ejecutar:
# guarda GROQ_API_KEY en Colab Secrets y habilita su acceso.

!pip install groq -q

from groq import Groq

client = Groq(
    api_key=userdata.get('GROQ_API_KEY')
)

def asistente(pregunta):

    # 1) Recuperar el fragmento más relevante.
    fragmento = buscar_fragmento(pregunta)

    # 2) Construir el prompt enriquecido con ese contexto.
    prompt = f"""Responde la pregunta del alumno usando SOLO el siguiente reglamento de la biblioteca. Si el reglamento no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

    # 3) Enviar el prompt al modelo generativo.
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # 4) Devolver únicamente el texto de la respuesta.
    return response.choices[0].message.content


# ============================================================
# PRUEBAS DEL PIPELINE
# ============================================================
#
# Probamos tres preguntas diferentes.
#
# La meta no es solo que "corra":
# observa si el retrieval selecciona información adecuada
# y si la respuesta permanece anclada al reglamento.

for pregunta in [
    "¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?",
    "¿Cuánto es la multa por entregar un libro tarde?",
    "¿Puedo renovar el préstamo de un libro?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {asistente(pregunta)}\n")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 13.7 MB/s eta 0:00:00
Pregunta: ¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?
Respuesta: No, solo se consultan dentro de la biblioteca.

Pregunta: ¿Cuánto es la multa por entregar un libro tarde?
Respuesta: $10 por día, máximo $150 por libro.

Pregunta: ¿Puedo renovar el préstamo de un libro?
Respuesta: Sí, una vez si nadie más lo ha solicitado.

